# Preprocessing and Filtering

There are a variety of ways to preprocess and filter data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import piprkit

## Generate a noisy example signal

As an example, we generate some noisy data.

In [ ]:
noisy_signal = piprkit.simulate_pupil_measurement(noise_std=0.001, trend_std=0.001, rng=0)

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
noisy_signal.plot(ax=ax, color="orange", linewidth=1.5, label="Moderate noise")
noisy_signal.plot_light_stimulus(ax=ax, color="yellow", alpha=0.3)
ax.legend()
plt.show()

## Apply simple smoothing filters

A rolling mean or median can be used to quickly smoothen the data. Using `time_window` the strength can be adjusted.

In [ ]:
# Let's apply a few filters to the noisy signal
rolling_mean_signal = noisy_signal.filters.rolling_mean(time_window=1, inplace=False)
rolling_median_signal = noisy_signal.filters.rolling_median(time_window=1, inplace=False)

fig, ax = plt.subplots(2, 1, figsize=(8, 4))
noisy_signal.plot(ax=ax[0], color="orange", linewidth=1.5, label="Moderate noise")
rolling_mean_signal.plot(ax=ax[0], color="blue", linewidth=1.5, label="Rolling mean")
noisy_signal.plot(ax=ax[1], color="orange", linewidth=1.5, label="Moderate noise")
rolling_median_signal.plot(ax=ax[1], color="green", linewidth=1.5, label="Rolling median")

noisy_signal.plot_light_stimulus(ax=ax[0], color="yellow", alpha=0.3)
noisy_signal.plot_light_stimulus(ax=ax[1], color="yellow", alpha=0.3)
ax[0].legend()
ax[1].legend()
plt.show()

## Add custom blinks

As an example, we add some blinks to the data.

In [ ]:
# Now let's add three blinks to the noisy data.
blink_signal = noisy_signal.copy()
blink_size = blink_signal.get_size().copy()

# Create a blink by replacing part of the signal with a quadratic function
size_during_blink_1 = (
    np.square(np.linspace(-1, 1, 50)) / 10 + np.mean(blink_size[150:200]) - 0.1
)
blink_size[150:200] = size_during_blink_1
size_during_blink_2 = (
    np.square(np.linspace(-2, 2, 20)) / 10 + np.mean(blink_size[350:370]) - 0.4
)
blink_size[350:370] = size_during_blink_2
size_during_blink_3 = (
    np.square(np.linspace(-1, 1, 15)) / 10 + np.mean(blink_size[600:615]) - 0.1
)
blink_size[600:615] = size_during_blink_3
blink_signal.set_time_and_size(blink_signal.get_time(), blink_size)

fig, ax = plt.subplots(2, 1, figsize=(8, 4))
noisy_signal.plot(ax=ax[0], color="orange", linewidth=1.5, label="Moderate noise")
blink_signal.plot(ax=ax[1], color="red", linewidth=1.5, label="With blink")

noisy_signal.plot_light_stimulus(ax=ax[0], color="yellow", alpha=0.3)
blink_signal.plot_light_stimulus(ax=ax[1], color="yellow", alpha=0.3)
ax[0].legend()
ax[1].legend()
plt.show()

## Smoothen blinks

The median and mean filters can be used to reduce the impact of blinks. Often, removal is a better strategy where possible.

In [ ]:
# Now we can try to filter the blinks using the mean and median filters again
blink_rolling_mean_signal = blink_signal.filters.rolling_mean(time_window=1, inplace=False)
blink_rolling_median_signal = blink_signal.filters.rolling_median(time_window=1, inplace=False)

fig, ax = plt.subplots(3, 1, figsize=(8, 6))
blink_signal.plot(ax=ax[0], color="red", linewidth=1.5, label="With blink")
blink_rolling_mean_signal.plot(ax=ax[1], color="blue", linewidth=1.5, label="Rolling mean")
blink_rolling_median_signal.plot(ax=ax[2], color="green", linewidth=1.5, label="Rolling median")

blink_signal.plot_light_stimulus(ax=ax[0], color="yellow", alpha=0.3)
blink_rolling_mean_signal.plot_light_stimulus(ax=ax[1], color="yellow", alpha=0.3)
blink_rolling_median_signal.plot_light_stimulus(ax=ax[2], color="yellow", alpha=0.3)

ax[0].legend()
ax[1].legend()
ax[2].legend()
plt.show()

## Blink removal

Blink removal where possible is generally more effective. Using `set_blinks` they can be set from a list obtained from your pupillometer (or other blink detection methods). These blinks can be plotted using `plot_blinks` or removed using `filters.remove_blinks`. `interpolate` can be used for interpolation.

In [ ]:
# If filtering is not sufficient, we can also remove the blinks.
time = blink_signal.get_time().copy()
print(f"Blink 1: Start at {time[150]:.1f} s, end at {time[200]:.1f} s")
print(f"Blink 2: Start at {time[350]:.1f} s, end at {time[370]:.1f} s")
print(f"Blink 3: Start at {time[600]:.1f} s, end at {time[615]:.1f} s")
blink_list = [(-7, -6), (-3, -2.6), (2, 2.3)]

marked_signal = blink_signal.copy()
marked_signal.set_blinks(blink_list)
blink_removed_signal = marked_signal.filters.remove_blinks(buffer_seconds=0.1, inplace=False)

interpolated_blinks_signal = blink_removed_signal.drop_nan(inplace=False)
interpolated_blinks_signal.interpolate(time)

fig, ax = plt.subplots(3, 1, figsize=(8, 6))
marked_signal.plot(ax=ax[0], color="red", linewidth=1.5, label="With blink")
blink_removed_signal.plot(ax=ax[1], color="purple", linewidth=1.5, label="Blinks removed")
interpolated_blinks_signal.plot(ax=ax[2], color="cyan", linewidth=1.5, label="Blinks interpolated")

marked_signal.plot_blinks(ax=ax[0], color="grey")
blink_removed_signal.plot_blinks(ax=ax[1], color="grey")
interpolated_blinks_signal.plot_blinks(ax=ax[2], color="grey")

marked_signal.plot_light_stimulus(ax=ax[0], color="yellow", alpha=0.3)
blink_removed_signal.plot_light_stimulus(ax=ax[1], color="yellow", alpha=0.3)
interpolated_blinks_signal.plot_light_stimulus(ax=ax[2], color="yellow", alpha=0.3)
ax[0].legend()
ax[1].legend()
ax[2].legend()
plt.show()

## Trim by pupil size

Artefacts can still exist; either due to missed blinks or due to other reasons. The data can be limited to fit between specific sizes to remove these artefacts. This can be done over the entire data, or over specific sections.

In [ ]:
# Another way to remove blinks or other outliers is by trimming the size.
trimmed_signal = blink_signal.copy()

# Removing the second blink
trimmed_signal.trim_size(min_size=0.9, max_size=1.2, start_time=-10, end_time=-1.5)
# Removing the third blink
trimmed_signal.trim_size(min_size=0.8, max_size=1.0, start_time=2.0, end_time=3.0)

fig, ax = plt.subplots(2, 1, figsize=(8, 4))
blink_signal.plot(ax=ax[0], color="red", linewidth=1.5, label="With blink")
trimmed_signal.plot(ax=ax[1], color="brown", linewidth=1.5, label="Trimmed size")

blink_signal.plot_light_stimulus(ax=ax[0], color="yellow", alpha=0.3)
trimmed_signal.plot_light_stimulus(ax=ax[1], color="yellow", alpha=0.3)
ax[0].legend()
ax[1].legend()
plt.show()

## Limit fast rates of change

Another method is to limit the rate of change using `filters.limit_rate_of_change`.

In [ ]:
# Finally, we can remove blinks by limiting the rate of change.
rate_limited_signal = blink_signal.copy()
# We don't limit the rate of change between -1 s and 0 s to preserve the constriction response to the light stimulus.
rate_limited_signal.filters.limit_rate_of_change(
    max_rate_of_change=0.1, start_time=-10, end_time=-1
)
rate_limited_signal.filters.limit_rate_of_change(
    max_rate_of_change=0.1, start_time=1, end_time=10
)

fig, ax = plt.subplots(2, 1, figsize=(8, 4))
blink_signal.plot(ax=ax[0], color="red", linewidth=1.5, label="With blink")
rate_limited_signal.plot(ax=ax[1], color="magenta", linewidth=1.5, label="Rate limited")
blink_signal.plot_light_stimulus(ax=ax[0], color="yellow", alpha=0.3)
rate_limited_signal.plot_light_stimulus(ax=ax[1], color="yellow", alpha=0.3)
ax[0].legend()
ax[1].legend()
plt.show()